# Can we predict 12-month loan default?

**Question:** using application-time attributes, can we predict `defaulted`?

**Plan:** data quality, EDA, baseline, logistic regression and gradient boosting with cross-validation, held-out test evaluation, then lift, calibration and importance.

The base rate is ~10%, so accuracy is misleading. We use ROC-AUC and PR-AUC (compared against the base rate), plus a lift view.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED=42
df=pd.read_csv('data.csv'); print(df.shape); pd.read_csv('columns.csv')

## 1. Data quality

In [ ]:
print('missing:', df.isna().sum().sum(), '| duplicate rows:', df.duplicated().sum(), '| duplicate ids:', df.application_id.duplicated().sum())
print('default rate: %.3f' % df.defaulted.mean())
df.describe().T

`application_id` is an identifier, so it is excluded from the features. There is no date column, so we use a random stratified split rather than a time-based one. If the real use is scoring future applications, a time-based validation would be a safer check.

## 2. Exploratory analysis

In [ ]:
num=['income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
fig,ax=plt.subplots(2,3,figsize=(13,7))
for a,col in zip(ax.ravel(),num):
    q=pd.qcut(df[col],10,duplicates='drop')
    r=df.groupby(q,observed=True).defaulted.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(df.defaulted.mean(),ls='--',c='grey')
    a.set_title(f'Default rate by {col} decile'); a.set_xlabel('decile (low to high)')
plt.tight_layout(); plt.show()
df.groupby('region').defaulted.agg(['mean','size'])

## 3. Split and models
Hold out 25% as a test set, used once at the end. Model comparison uses 5-fold stratified CV on the training portion.

In [ ]:
features=['region']+num
X,y=df[features],df.defaulted
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=SEED)
pre=ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['region']),('n',StandardScaler(),num)])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic regression':Pipeline([('p',pre),('m',LogisticRegression(max_iter=1000))]),
 'gradient boosting':Pipeline([('p',ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['region']),('n','passthrough',num)])),
                               ('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
    s=cross_validate(m,Xtr,ytr,cv=cv,scoring=['roc_auc','average_precision'])
    rows.append(dict(model=n,auc=s['test_roc_auc'].mean(),auc_sd=s['test_roc_auc'].std(),pr_auc=s['test_average_precision'].mean()))
pd.DataFrame(rows).round(4)

## 4. Held-out test performance

In [ ]:
fitted={n:m.fit(Xtr,ytr) for n,m in models.items()}
P={n:m.predict_proba(Xte)[:,1] for n,m in fitted.items()}
base=yte.mean()
res=pd.DataFrame({n:dict(auc=roc_auc_score(yte,p),pr_auc=average_precision_score(yte,p),brier=brier_score_loss(yte,p)) for n,p in P.items()}).T
print('test base rate: %.3f'%base); display(res.round(4))
best=res.auc.drop('baseline (prior)').idxmax(); rng=np.random.default_rng(SEED); yt=yte.values; pb=P[best]
bs=[roc_auc_score(yt[i],pb[i]) for i in (rng.integers(0,len(yt),len(yt)) for _ in range(500))]
print(best,'AUC 95%% CI: %.3f-%.3f'%tuple(np.percentile(bs,[2.5,97.5])))
fig,ax=plt.subplots(1,2,figsize=(11,4))
for n,p in P.items():
    if n.startswith('baseline'): continue
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set(title='ROC',xlabel='FPR',ylabel='TPR'); ax[1].axhline(base,c='k',ls='--'); ax[1].set(title='Precision-recall',xlabel='recall',ylabel='precision')
ax[0].legend(); plt.tight_layout(); plt.show()

## 5. Practical value: lift by risk decile and calibration

In [ ]:
p=P[best]; t=pd.DataFrame({'p':p,'y':yte.values}); t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=False)+1
lift=t.groupby('decile').agg(pred=('p','mean'),actual=('y','mean'),n=('y','size'))
display(lift.round(3))
print('Top-decile default rate %.1f%% vs base %.1f%% (lift %.1fx); riskiest 30%% captures %.0f%% of defaults'%(
  100*lift.actual.iloc[-1],100*base,lift.actual.iloc[-1]/base,100*t[t.decile>=8].y.sum()/t.y.sum()))
fr,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile')
plt.plot(mp,fr,'o-'); plt.plot([0,.4],[0,.4],'k--'); plt.xlabel('mean predicted'); plt.ylabel('observed'); plt.title('Calibration ('+best+')'); plt.show()

## 6. What drives the prediction?

In [ ]:
pi=permutation_importance(fitted[best],Xte,yte,scoring='roc_auc',n_repeats=20,random_state=SEED)
imp=pd.Series(pi.importances_mean,index=features); sd=pd.Series(pi.importances_std,index=features)
order=imp.sort_values().index; imp[order].plot.barh(xerr=sd[order]); plt.xlabel('AUC drop when shuffled'); plt.show()
lr=fitted['logistic regression']
pd.Series(lr[-1].coef_[0],index=lr[:-1].get_feature_names_out()).sort_values().round(3)